# Chapter 16 &mdash; Both Normal Forms, Read Off One Diagram

**Concept 14 of the Chapter 16 decomposition:** *Both Normal Forms, Read Off One Diagram*

Paths to the 1 node spell a DNF; paths to the 0 node spell a CNF, by De Morgan. One structure, both forms.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-Normal-Forms-From-BDD/Concept-Normal-Forms-From-BDD.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Bdd            import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Every root-to-terminal path in a BDD is a **cube**: the conjunction of the literals on
its edges. The variables the path skipped are **don't-cares** &mdash; it skipped them
precisely because they did not matter along that path.

So the diagram hands you both normal forms:

* the paths reaching **1** are the products of a **DNF**;
* the paths reaching **0** are the ways to make the formula *false*, so negating the
  literals on each gives one **clause**, and together a **CNF**.

The second reading is De Morgan doing the work, and it is why the `0` terminal is worth
as much as the `1` terminal.

Two degenerate cases carry the idea. The constant-`0` diagram has **no** path to `1`,
so its DNF is `0`. The constant-`1` diagram reaches `1` by a path carrying **no
literals at all** &mdash; an empty product, which is *true*. The mirror image holds for
CNF, where an empty **clause** is *false*.

## 2. Definitions

### First, the diagram &mdash; everything below is read off it

In [ ]:
f = bdd('''
Var_Order : a b c d
f = a | (b & c & d)
Main_Exp : f
''')
f

### The paths are the formula

Follow each route from the root. Blue edges are 1, red are 0; the variables a route skips are the ones that do not matter along it.

In [ ]:
print('paths to 1:')
for p in paths(f, 1):
    print('   ', p)
print()
print('paths to 0:')
for p in paths(f, 0):
    print('   ', p)

<!-- nav-strip -->

---

&larr;&nbsp;[Ch16&nbsp;13.&nbsp;A BDD Decides SAT by Being Built](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-BDD-Decides-SAT/Concept-BDD-Decides-SAT.ipynb) &nbsp;&middot;&nbsp; [**Chapter 16** index](https://github.com/ganeshutah/Jove/blob/master/Chapter16/README.md) &nbsp;&middot;&nbsp; [Ch16&nbsp;15.&nbsp;Why Converting CNF to DNF Is Not a Free Lunch](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter16/Concept-CNF-To-DNF-Is-No-Free-Lunch/Concept-CNF-To-DNF-Is-No-Free-Lunch.ipynb)&nbsp;&rarr;

---

## 3. Tests

Joined up, those are the two normal forms.

In [ ]:
print('DNF:', dnf(f))
print('CNF:', cnf(f))

**Check them**, rather than believe them. Evaluate both forms at every assignment and compare against the diagram.

In [ ]:
from itertools import product as _prod

def ev(expr, env):
    if expr in ('0', '1'):
        return bool(int(expr))
    py = expr.replace('~', ' not ').replace('&', ' and ').replace('|', ' or ')
    return bool(eval(py, {}, {k: bool(v) for k, v in env.items()}))

truth = {tuple(sorted(m.items())) for m in f.models}
D, C = dnf(f), cnf(f)
for combo in _prod([0, 1], repeat=len(f.vars)):
    env = dict(zip(f.vars, combo))
    want = tuple(sorted(env.items())) in truth
    assert ev(D, env) == want, ('DNF disagrees at', env)
    assert ev(C, env) == want, ('CNF disagrees at', env)
print('DNF and CNF agree with the diagram on all %d assignments.'
      % 2 ** len(f.vars))

**The degenerate cases.** An empty product is true; an empty clause is false. Both fall out of the same walk.

In [ ]:
u = bdd('Var_Order : p\nMain_Exp : p & ~p')
t = bdd('Var_Order : p\nMain_Exp : p | ~p')
print('unsatisfiable  DNF: %-4s CNF: %s' % (dnf(u), cnf(u)))
print('tautology      DNF: %-4s CNF: %s' % (dnf(t), cnf(t)))
print()
print('paths to 1 from the tautology:', paths(t, 1), '<- one path, no literals')
assert dnf(u) == '0' and cnf(t) == '1'

## 4. Exercises


1. `a <=> b` has a four-node diagram. Write out its DNF and CNF by hand from the
   picture, then check against `dnf` and `cnf`.
2. The DNF above has two terms, while `f.models` has nine entries. Which variables
   were don't-cares on which path, and how does that account for the gap?
3. Negate a formula by swapping the roles of the terminals: predict `dnf` of the
   negation from `cnf` of the original, then check.
4. Build a formula whose DNF is shorter than its CNF, and one where the reverse
   holds. What in the diagram decides which way it goes?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter16/Concept-Normal-Forms-From-BDD')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')